# CodeTune v3 Colab Training

This notebook is for Colab Pro A100/H100 runs.

What it does:
- mounts Google Drive
- installs training dependencies
- clones or unpacks the repo
- syncs prepared data into the Colab workspace
- picks the A100 or H100 config automatically
- runs SFT and optional DPO steps
- syncs checkpoints back to Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, torch
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('BF16 supported:', torch.cuda.is_bf16_supported())


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Total VRAM: 85.1 GB
BF16 supported: True


In [ ]:
# Required repo source: set either REPO_URL or REPO_ARCHIVE_PATH.
REPO_URL = 'https://github.com/MichLitt/coding-llm-finetune'
REPO_BRANCH = 'main'
REPO_ARCHIVE_PATH = ''  # zip/tar.gz in Google Drive, if you do not want to clone

# Google Drive persistence.
DRIVE_ROOT = '/content/drive/MyDrive/coding-llm-colab'
DATA_SOURCE_DIR = '/content/drive/MyDrive/coding-llm-data/processed'

# Auth.
HF_TOKEN = ''
WANDB_API_KEY = ''

# Run switches.
RUN_SFT = True
RUN_DPO_PAIR_GEN = False
RUN_DPO = False

# Experiment params.
SFT_EXP_ID = 'sft_targeted'
DPO_EXP_ID = 'dpo_v1'
SFT_RESUME_FROM = ''
SFT_CHECKPOINT = ''  # leave blank to use the SFT final adapter after training
MBPP_NUM_PROBLEMS = 250
MBPP_N_CANDIDATES = 8

WORKDIR = '/content/Coding-LLM'


In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q trl peft accelerate bitsandbytes datasets huggingface-hub python-dotenv pyyaml click wandb


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 162.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 25.3 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

workdir = Path(WORKDIR)
if workdir.exists():
    shutil.rmtree(workdir)

if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(workdir)], check=True)
elif REPO_ARCHIVE_PATH:
    archive_path = Path(REPO_ARCHIVE_PATH)
    if not archive_path.exists():
        raise FileNotFoundError(f'Repo archive not found: {archive_path}')
    workdir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archive_path), str(workdir))
    children = list(workdir.iterdir())
    if len(children) == 1 and children[0].is_dir():
        extracted_root = children[0]
        for child in extracted_root.iterdir():
            shutil.move(str(child), workdir / child.name)
        extracted_root.rmdir()
else:
    raise ValueError('Set REPO_URL or REPO_ARCHIVE_PATH before running this cell.')

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
(drive_root / 'logs').mkdir(parents=True, exist_ok=True)
(drive_root / 'artifacts').mkdir(parents=True, exist_ok=True)

processed_dst = workdir / 'data' / 'processed'
processed_dst.mkdir(parents=True, exist_ok=True)
if DATA_SOURCE_DIR:
    data_src = Path(DATA_SOURCE_DIR)
    if not data_src.exists():
        raise FileNotFoundError(f'Data source dir not found: {data_src}')
    subprocess.run(['bash', '-lc', f"rsync -a '{data_src}/' '{processed_dst}/'"], check=True)

env_lines = []
if HF_TOKEN:
    env_lines.append(f'HF_TOKEN={HF_TOKEN}')
if WANDB_API_KEY:
    env_lines.append(f'WANDB_API_KEY={WANDB_API_KEY}')
if env_lines:
    (workdir / '.env').write_text('\n'.join(env_lines) + '\n', encoding='utf-8')

print('Workspace ready:', workdir)


In [ ]:
from pathlib import Path
import torch

gpu_name = torch.cuda.get_device_name(0).upper()
profile = 'h100' if 'H100' in gpu_name else 'a100'
SFT_CONFIG_PATH = f'configs/sft_config_colab_{profile}.yaml'
DPO_CONFIG_PATH = f'configs/dpo_config_colab_{profile}.yaml'

print('Selected profile:', profile)
print('SFT config:', SFT_CONFIG_PATH)
print('DPO config:', DPO_CONFIG_PATH)
print('Train data exists:', (Path(WORKDIR) / 'data/processed/sft_train.jsonl').exists())


In [ ]:
import subprocess
from pathlib import Path

def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=WORKDIR)

def sync_back():
    subprocess.run(
        ['bash', '-lc', f"rsync -a '{WORKDIR}/results/' '{DRIVE_ROOT}/artifacts/results/'"],
        check=True,
    )
    if (Path(WORKDIR) / 'wandb').exists():
        subprocess.run(
            ['bash', '-lc', f"rsync -a '{WORKDIR}/wandb/' '{DRIVE_ROOT}/artifacts/wandb/'"],
            check=True,
        )

if RUN_SFT:
    cmd = [
        'python',
        'scripts/sft_train.py',
        '--config-path', SFT_CONFIG_PATH,
        '--exp-id', SFT_EXP_ID,
    ]
    if SFT_RESUME_FROM:
        cmd += ['--resume-from', SFT_RESUME_FROM]
    run(cmd)
    sync_back()

sft_final_dir = Path(WORKDIR) / 'results' / 'sft_checkpoints' / SFT_EXP_ID / 'final'
if not SFT_CHECKPOINT:
    SFT_CHECKPOINT = str(sft_final_dir)

print('Active SFT checkpoint:', SFT_CHECKPOINT)


In [ ]:
if RUN_DPO_PAIR_GEN:
    run([
        'python',
        'scripts/generate_dpo_pairs.py',
        '--sft-checkpoint', SFT_CHECKPOINT,
        '--num-problems', str(MBPP_NUM_PROBLEMS),
        '--n-candidates', str(MBPP_N_CANDIDATES),
    ])
    sync_back()

if RUN_DPO:
    run([
        'python',
        'scripts/dpo_train.py',
        '--config-path', DPO_CONFIG_PATH,
        '--sft-checkpoint', SFT_CHECKPOINT,
        '--exp-id', DPO_EXP_ID,
    ])
    sync_back()
